In [13]:
# Step 1: Install required packages (if not already installed)
#!pip install langchain langchain-google-genai langgraph requests python-dotenv

In [14]:
# Step 2: Import required libraries
import os
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Load API key from .env file
load_dotenv()

True

In [15]:
# Step 3: Define REST API URLs
PRODUCT_API = "http://127.0.0.1:8000/products"
ORDER_API = "http://127.0.0.1:8002/orders"

In [16]:
# Step 4: Create Tool to fetch products from Product API
@tool
def get_products():
    """Get all product details from the Product API."""
    try:
        response = requests.get(PRODUCT_API, timeout=5)
        return response.json()
    except Exception as e:
        return f"Error connecting to Product API: {e}"

In [17]:
# Step 5: Create Tool to fetch orders from Order API
@tool
def get_orders():
    """Get all order details from the Order API."""
    try:
        response = requests.get(ORDER_API, timeout=5)
        return response.json()
    except Exception as e:
        return f"Error connecting to Order API: {e}"

In [18]:
# Step 6: Initialize Gemini Model and Tools list
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

tools = [get_products, get_orders]

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [19]:
# Step 7: Create Agent using LangGraph
agent = create_react_agent(
    model=llm,
    tools=tools
)

C:\Users\admin\AppData\Local\Temp\ipykernel_388\3871573776.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [20]:
# Step 8: Define System Prompt (Strictly answer only products and orders)
system_message = """
You are a product and order chatbot.

You can ONLY answer questions about:
1. Products
2. Orders

For product questions, get the information from the Product API.
For order questions, get the information from the Order API.

Never make up product or order information.

If the user asks anything unrelated to products or orders, say:
"I can only help with product and order information."
"""

In [21]:
# Step 9: Test 1 - Ask about products
question = "What products are available?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

We have the following products available:

*   **iPhone 15**: Apple smartphone with 128GB storage, priced at 69999, with 10 units in stock.
*   **MacBook Air M2**: 13.6-inch Liquid Retina Display, 8GB RAM, 256GB SSD, priced at 99999, with 5 units in stock.


In [22]:
# Step 10: Test 2 - Ask about specific product details
question = "What is the price of iPhone 15 and is it in stock?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': 'The price of iPhone 15 is 69999 and it is in stock with 10 units available.', 'extras': {'signature': 'CqcFARFNMg/O+iB+O30qG2Q5X63XyAPNp910eb5qz0JsCXX0fRdymTS0h5l7R9bychxUDmxwJ+lwPGsLmz8NNl7wuaIjYFmmxjhrET+4/4xEPqNIc6ySGZLI53p7x42oqQUMvgwGGxIjHW1OvIRWsTT5rnK7TX4GLxwFPlnkWCNLtKVvhYAjXvE6PsWUtphCI2N8lNnMhWPTCmcxO0f/hUmqqQ9zMj1L3UI45h4me/0zcRpOp8GrU/D4U6PMeeGJagmxNOWpsYxL7HrL96S2g06QtsI+Uc0Ywbnb1vDTMRHi0R5RG5UikSeflv/E7bB4mqhUnejrXCqgfpQAqpRGJa3mlcMEDe0rysog0cEFGF3tmgt9He8tyEa/4Dq8kvJpI1TPJMOtz+6piLmReRl2KW3YvUxS5xcbumTx20cSTvK0OV3txvQwILGph0j2sZi/bLa0R0HOAQIV3W/tmp2NNODrvpoxG1vt3p8MpJpzw7iWoeyTqg2i/bNP7lBCkftesei13x+X7Xvp3JxY98DYrqagBqfjpTH/uU8BwomxMHlh3kVvVPaMMbkuZHJCnCPdz1P4fcbJe63ik844kuxHhRkPZutLKo4Bi9iz89EqT1x84noIDtGdkyMUCFgDBTudAhGqPXGxJVUSqwlADozr4A0LxsTYGM/mqvsOyNXlKAUnCC9lm4GphSsenziBgMmgAgvhak8PusmA+kXS/Q78XTl9XHoNUm7FG79at8SZAjZVPm1ZxN6LDVE3BoKb1s4J1CIzHSVQfrTgO2yqo+qHssRe5P1g5NID7kWKsah6uDESo49jFMj+JXh6tWoHTQB2i4dtT+MWmDfsdU/Ikhq65haoHo2o/iQEQbcm0B

In [23]:
# Step 11: Test 3 - Ask about order status
question = "What is the status of order 101?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': 'Order 101 is Shipped.', 'extras': {'signature': 'CuIBARFNMg+ahTRLrCOhNAkwjGcGuZVaZ4MEpc71jJ7S1Mztvaf9Rj93J/JkQunRLq9ZehKZMqtx+VF0KQvlLg/WzDaZyMl/P8z3WZJinGaq3BRZEOnk0l7Y5T4duGgytks8gCS0gK5r+LKJGFlpqWTvttA+E/V0Q4f9K2JrhYilwKlzAT6//exBWLzLHRtecZwUJjqoyDiBX9dBws7jPNjpZJ1+KyB0gNNGvTttvdjT+XnP1pvIPbrd1b2SAa3SqZ0bO2yis0K7AatAfk9D7/+F9ZLMVhjA1Br0eBDeWGlg4YERTA=='}}]


In [24]:
# Step 12: Test 4 - Ask unrelated question (must be refused)
question = "What is the capital of India?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

I can only help with product and order information.
